# Assignment Beta - Question 3
### Knowledge Distillation: one training step by hand
Ibrahim Johar Farooqi - 23K-0074 - BAI-7A

Prompt: *"The patient was admitted to the ___"*, vocab = [hospital, clinic, ward, school], correct word is hospital.

Loss used for training:

$$L = \alpha \cdot T^2 \cdot KL(p_T \,\|\, q_T) + (1-\alpha)\cdot CE(y, q_1)$$

with $T = 2$, $\alpha = 0.5$, natural log everywhere. Everything is done with plain numpy so each step can be printed and checked against the hand working in the report.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)

words = ["hospital", "clinic", "ward", "school"]
z_t = np.array([5.0, 3.0, 2.0, 0.0])
z_s = np.array([3.0, 2.5, 1.0, 0.5])
y = np.array([1.0, 0.0, 0.0, 0.0])

T = 2
alpha = 0.5
lr = 0.5

In [2]:
def softmax(z, T=1.0):
    e = np.exp(z / T)
    return e / e.sum()

def show_softmax(name, z, T):
    e = np.exp(z / T)
    print(f"{name}  z/T = {z / T}")
    print(f"{' ' * len(name)}  exp = {e}   sum = {e.sum():.4f}")
    print(f"{' ' * len(name)}  p   = {e / e.sum()}")

## (a) Probabilities at T = 1

In [3]:
show_softmax("teacher", z_t, 1)
print()
show_softmax("student", z_s, 1)

p1 = softmax(z_t, 1)
q1 = softmax(z_s, 1)

teacher  z/T = [5. 3. 2. 0.]
         exp = [148.4132  20.0855   7.3891   1.    ]   sum = 176.8878
         p   = [0.839  0.1135 0.0418 0.0057]

student  z/T = [3.  2.5 1.  0.5]
         exp = [20.0855 12.1825  2.7183  1.6487]   sum = 36.6350
         p   = [0.5483 0.3325 0.0742 0.045 ]


## (b) Softened probabilities at T = 2

In [4]:
show_softmax("teacher", z_t, T)
print()
show_softmax("student", z_s, T)

pT = softmax(z_t, T)
qT = softmax(z_s, T)

pd.DataFrame({"teacher T=1": p1, "student T=1": q1, "teacher T=2": pT, "student T=2": qT}, index=words).round(4)

teacher  z/T = [2.5 1.5 1.  0. ]
         exp = [12.1825  4.4817  2.7183  1.    ]   sum = 20.3825
         p   = [0.5977 0.2199 0.1334 0.0491]

student  z/T = [1.5  1.25 0.5  0.25]
         exp = [4.4817 3.4903 1.6487 1.284 ]   sum = 10.9048
         p   = [0.411  0.3201 0.1512 0.1177]


,teacher T=1,student T=1,teacher T=2,student T=2
hospital,0.8390,0.5483,0.5977,0.4110
clinic,0.1135,0.3325,0.2199,0.3201
ward,0.0418,0.0742,0.1334,0.1512
school,0.0057,0.0450,0.0491,0.1177


## (c) KL divergence between the softened distributions

$KL(p_T \| q_T) = \sum_i p_{T,i} \ln \dfrac{p_{T,i}}{q_{T,i}}$

In [5]:
kl_terms = pT * np.log(pT / qT)
for w, p, q, t in zip(words, pT, qT, kl_terms):
    print(f"{w:9s} {p:.4f} * ln({p:.4f}/{q:.4f}) = {t:+.4f}")

kl = kl_terms.sum()
print("KL =", round(kl, 4))

hospital  0.5977 * ln(0.5977/0.4110) = +0.2239
clinic    0.2199 * ln(0.2199/0.3201) = -0.0826
ward      0.1334 * ln(0.1334/0.1512) = -0.0167
school    0.0491 * ln(0.0491/0.1177) = -0.0430
KL = 0.0816


## (d) Hard-label cross-entropy
Only the hospital term survives since y is one-hot.

In [6]:
ce = -np.sum(y * np.log(q1))
print(f"CE = -ln({q1[0]:.4f}) = {ce:.4f}")

CE = -ln(0.5483) = 0.6010


## (e) Total distillation loss

In [7]:
soft = alpha * T**2 * kl
hard = (1 - alpha) * ce
loss = soft + hard

print(f"soft term = {alpha} * {T**2} * {kl:.4f} = {soft:.4f}")
print(f"hard term = {1 - alpha} * {ce:.4f} = {hard:.4f}")
print(f"L = {loss:.4f}")

soft term = 0.5 * 4 * 0.0816 = 0.1632
hard term = 0.5 * 0.6010 = 0.3005
L = 0.4637


## (f) Gradient w.r.t. student logits

$\dfrac{\partial CE}{\partial z_s} = q_1 - y$ and $\dfrac{\partial KL}{\partial z_s} = \dfrac{1}{T}(q_T - p_T)$, so

$$\frac{\partial L}{\partial z_s} = \alpha T (q_T - p_T) + (1-\alpha)(q_1 - y)$$

In [8]:
grad_soft = alpha * T * (qT - pT)
grad_hard = (1 - alpha) * (q1 - y)
grad = grad_soft + grad_hard

pd.DataFrame({"qT - pT": qT - pT, "soft part": grad_soft, "q1 - y": q1 - y, "hard part": grad_hard, "dL/dz": grad}, index=words).round(4)

,qT - pT,soft part,q1 - y,hard part,dL/dz
hospital,-0.1867,-0.1867,-0.4517,-0.2259,-0.4126
clinic,0.1002,0.1002,0.3325,0.1663,0.2665
ward,0.0178,0.0178,0.0742,0.0371,0.0549
school,0.0687,0.0687,0.0450,0.0225,0.0912


In [9]:
def total_loss(zs):
    qT_ = softmax(zs, T)
    q1_ = softmax(zs, 1)
    return alpha * T**2 * np.sum(pT * np.log(pT / qT_)) - (1 - alpha) * np.log(q1_[0])

# finite difference check on the analytic gradient
eps = 1e-6
num_grad = np.array([(total_loss(z_s + eps * np.eye(4)[i]) - total_loss(z_s - eps * np.eye(4)[i])) / (2 * eps) for i in range(4)])
print("analytic:", grad)
print("numeric: ", num_grad)
print("gradient sums to", round(grad.sum(), 10))

analytic: [-0.4126  0.2665  0.0549  0.0912]
numeric:  [-0.4126  0.2665  0.0549  0.0912]
gradient sums to -0.0


## (g) One gradient descent step, η = 0.5

In [10]:
z_new = z_s - lr * grad
print("new student logits:", z_new)

q1_new = softmax(z_new, 1)
qT_new = softmax(z_new, T)
kl_new = np.sum(pT * np.log(pT / qT_new))
ce_new = -np.log(q1_new[0])
loss_new = alpha * T**2 * kl_new + (1 - alpha) * ce_new

print("q1 new:", q1_new)
print("qT new:", qT_new)
print(f"KL = {kl_new:.4f}, CE = {ce_new:.4f}")
print(f"loss after step = {loss_new:.4f}  (was {loss:.4f}, drop of {loss - loss_new:.4f})")

new student logits: [3.2063 2.3668 0.9725 0.4544]
q1 new: [0.6239 0.2695 0.0668 0.0398]
qT new: [0.447  0.2938 0.1463 0.1129]
KL = 0.0567, CE = 0.4718
loss after step = 0.3493  (was 0.4637, drop of 0.1145)


In [11]:
pd.DataFrame({
    "before": [*z_s, q1[0], kl, ce, loss],
    "after": [*z_new, q1_new[0], kl_new, ce_new, loss_new],
}, index=[f"z_{w}" for w in words] + ["q1(hospital)", "KL", "CE", "L"]).round(4)

,before,after
z_hospital,3.0000,3.2063
z_clinic,2.5000,2.3668
z_ward,1.0000,0.9725
z_school,0.5000,0.4544
q1(hospital),0.5483,0.6239
KL,0.0816,0.0567
CE,0.6010,0.4718
L,0.4637,0.3493
